Databricks notebook: nb_gold_fact_charging_session
Source : ev_catalog.silver.sl_sessions  +  sl_payments  +  sl_tariffs
Sink   : ev_catalog.gold.fact_charging_session  (Delta, external)
Grain  : One row per charging session

Silver columns used:
  sl_sessions : session_id, vehicle_id, station_id, customer_id, started_at, ended_at,
                duration_min, energy_kwh, cost_aud, peak_power_kw, connector_type,
                session_status, payment_id, created_at, updated_at
  sl_payments : payment_id, session_id, customer_id, gateway, amount_aud, gst,
                payment_mode, status, processed_at
  sl_tariffs  : tariff_id, rate_per_kwh, is_current

Note: sl_sessions does not have tariff_id, charger_id, sla/gst flags.
      We join payments via payment_id from sessions.

In [ ]:
# ── Cell 1: imports and config ────────────────────────────────────────────────
from pyspark.sql import functions as F
from delta.tables import DeltaTable

GOLD_TABLE = "ev_catalog.gold.fact_charging_session"
GOLD_PATH  = "abfss://gold@evdatalakedev2.dfs.core.windows.net/facts/fact_charging_session/"

dbutils.widgets.text("p_run_date", "", "Run date YYYY-MM-DD (blank = full load)")
dbutils.widgets.text("p_run_type", "", "Run type full/incremental (blank = full load)")
run_date = dbutils.widgets.get("p_run_date").strip()
run_type = dbutils.widgets.get("p_run_type").strip()

In [ ]:
# ── Cell 2: read Silver sources ────────────────────────────────────────────────
df_sessions = spark.table("ev_catalog.silver.sl_sessions")
df_payments = spark.table("ev_catalog.silver.sl_payments")
df_tariffs  = spark.table("ev_catalog.silver.sl_tariffs")

# Incremental filter: sl_sessions has sl_batch_id from Silver MERGE
if run_date and run_type == 'incremental':
    df_sessions = df_sessions.filter(F.col("sl_batch_id") == run_date)
    print(f"Incremental: filtering sessions for batch_id = {run_date}")

In [ ]:
# ── Cell 3: resolve current tariff ────────────────────────────────────────────
# sl_tariffs: tariff_id, rate_per_kwh (string, cast to double), is_current (boolean)
# There is no tariff_id on sessions — we use the single current tariff rate as a
# global rate for expected-amount calculation.
df_current_tariff = (
    df_tariffs
    .filter(F.col("is_current") == True)
    .select(
        F.col("tariff_id"),
        F.col("rate_per_kwh").cast("double"),
    )
    .limit(1)  # take one current rate for expected-amount estimation
)

current_rate = df_current_tariff.collect()
rate_per_kwh = float(current_rate[0]["rate_per_kwh"]) if current_rate else 0.0
tariff_key   = current_rate[0]["tariff_id"]           if current_rate else "UNKNOWN"
print(f"Current tariff: {tariff_key}  rate={rate_per_kwh}")

In [ ]:
# ── Cell 4: build fact ────────────────────────────────────────────────────────
# Sessions join to payments via payment_id (sessions.payment_id = payments.payment_id).
# sl_sessions: started_at, ended_at (strings) — parse to timestamp.
# sl_payments: amount_aud, gst (strings) — cast to double.
# expected_amount = energy_kwh * current rate.

df_fact = (
    df_sessions.alias("s")
    .join(
        df_payments.alias("p").select(
            "payment_id", "session_id",
            F.col("amount_aud").cast("double").alias("amount_aud"),
            F.col("gst").cast("double").alias("gst"),
            "status", "gateway", "payment_mode"
        ),
        F.col("s.payment_id") == F.col("p.payment_id"),
        "left"
    )
    .select(
        # Keys
        F.col("s.session_id")                           .alias("session_key"),
        F.col("s.customer_id")                          .alias("customer_key"),
        F.col("s.station_id")                           .alias("station_key"),
        F.lit(tariff_key)                               .alias("tariff_key"),
        # Time
        F.date_format(
            F.to_timestamp(F.col("s.started_at")), "yyyyMMddHH"
        ).cast("long")                                  .alias("time_key"),
        F.to_date(F.col("s.started_at"))                .alias("session_date"),
        # Measures
        F.col("s.energy_kwh").cast("double"),
        F.col("s.duration_min").cast("int"),
        F.col("s.peak_power_kw").cast("double"),
        F.col("s.cost_aud").cast("double"),
        # Expected amount from current tariff rate
        F.round(
            F.col("s.energy_kwh").cast("double") * F.lit(rate_per_kwh), 2
        )                                               .alias("expected_amount_aud"),
        # Billed amount from payment
        F.col("p.amount_aud")                           .alias("billed_amount_aud"),
        F.col("p.gst")                                  .alias("gst_aud"),
        # Payment attributes
        F.col("p.status")                               .alias("payment_status"),
        F.col("p.gateway")                              .alias("payment_gateway"),
        F.col("p.payment_mode"),
        # Reconciliation: UNPAID if no payment, MATCH if within 1 cent, else MISMATCH
        F.when(
            F.col("p.amount_aud").isNull(), "UNPAID"
        ).when(
            F.abs(
                F.col("p.amount_aud") -
                F.round(F.col("s.energy_kwh").cast("double") * F.lit(rate_per_kwh), 2)
            ) < 0.01,
            "MATCH"
        ).otherwise("MISMATCH")                         .alias("reconciliation_status"),
        # Session status
        F.col("s.session_status"),
        F.col("s.connector_type"),
        # Metadata
        F.current_timestamp()                           .alias("gold_updated_at"),
        F.lit(run_date if run_date else "full")         .alias("gold_batch_id"),
    )
)

total = df_fact.count()
print(f"Fact rows: {total}")
if total == 0:
    print("No sessions to process. Exiting.")
    dbutils.notebook.exit("NO_DATA")

df_fact.show(3, truncate=80)

In [ ]:
# ── Cell 5: reconciliation summary (before write) ─────────────────────────────
df_fact.groupBy("reconciliation_status").count().display()

In [ ]:
# ── Cell 6: CREATE TABLE IF NOT EXISTS ────────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
    session_key           STRING    NOT NULL,
    customer_key          STRING,
    station_key           STRING,
    tariff_key            STRING,
    time_key              LONG,
    session_date          DATE,
    energy_kwh            DOUBLE,
    duration_min          INT,
    peak_power_kw         DOUBLE,
    cost_aud              DOUBLE,
    expected_amount_aud   DOUBLE,
    billed_amount_aud     DOUBLE,
    gst_aud               DOUBLE,
    payment_status        STRING,
    payment_gateway       STRING,
    payment_mode          STRING,
    reconciliation_status STRING,
    session_status        STRING,
    connector_type        STRING,
    gold_updated_at       TIMESTAMP,
    gold_batch_id         STRING
)
USING DELTA
LOCATION '{GOLD_PATH}'
""")

In [ ]:
# ── Cell 7: MERGE ─────────────────────────────────────────────────────────────
tgt = DeltaTable.forName(spark, GOLD_TABLE)
(
    tgt.alias("t")
       .merge(df_fact.alias("s"), "t.session_key = s.session_key")
       .whenMatchedUpdateAll()
       .whenNotMatchedInsertAll()
       .execute()
)
print(f"MERGE done: {GOLD_TABLE}")

In [ ]:
# ── Cell 8: post-load reconciliation breakdown ────────────────────────────────
spark.sql(f"""
SELECT
    reconciliation_status,
    COUNT(*)                               AS session_count,
    ROUND(SUM(expected_amount_aud), 2)     AS total_expected,
    ROUND(SUM(billed_amount_aud),   2)     AS total_billed,
    ROUND(SUM(billed_amount_aud) - SUM(expected_amount_aud), 2) AS variance_aud
FROM {GOLD_TABLE}
GROUP BY reconciliation_status
ORDER BY reconciliation_status
""").display()